# NYC Taxi Data Exploration
## FDE Assignment - Initial Data Analysis

This notebook demonstrates data profiling and exploration before building the pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Project paths
project_root = Path.cwd().parent
data_dir = project_root / 'data'
raw_dir = data_dir / 'raw'
output_dir = data_dir / 'output'

## Load Raw Data

In [ ]:
# Load zone lookup
zones_df = pd.read_csv(raw_dir / 'taxi_zone_lookup.csv')
print(f"Zones loaded: {len(zones_df)}")
zones_df.head()

In [ ]:
# Load trip data
trips_df = pd.read_parquet(raw_dir / 'yellow_tripdata_sample.parquet')
print(f"Trips loaded: {len(trips_df):,}")
trips_df.head()

## Data Profiling

In [ ]:
# Basic info
print("=" * 60)
print("DATASET INFO")
print("=" * 60)
trips_df.info()

In [ ]:
# Numeric statistics
trips_df.describe()

In [ ]:
# Missing values
missing = trips_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("\nMissing Values:")
print(missing)

## Data Quality Issues

In [ ]:
# Negative fares
negative_fares = trips_df[trips_df['fare_amount'] < 0]
print(f"Negative fares: {len(negative_fares)} ({len(negative_fares)/len(trips_df)*100:.2f}%)")

# Zero distance
zero_distance = trips_df[trips_df['trip_distance'] == 0]
print(f"Zero distance: {len(zero_distance)} ({len(zero_distance)/len(trips_df)*100:.2f}%)")

# Future dates
future_dates = trips_df[trips_df['tpep_pickup_datetime'] > pd.Timestamp.now()]
print(f"Future dates: {len(future_dates)} ({len(future_dates)/len(trips_df)*100:.2f}%)")

## Distributions

In [ ]:
# Trip distance distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Distance
valid_distance = trips_df[trips_df['trip_distance'] > 0]['trip_distance']
axes[0].hist(valid_distance[valid_distance < 20], bins=50, edgecolor='black')
axes[0].set_title('Trip Distance Distribution')
axes[0].set_xlabel('Distance (miles)')
axes[0].set_ylabel('Frequency')

# Fare amount
valid_fare = trips_df[trips_df['fare_amount'] > 0]['fare_amount']
axes[1].hist(valid_fare[valid_fare < 100], bins=50, edgecolor='black')
axes[1].set_title('Fare Amount Distribution')
axes[1].set_xlabel('Fare ($)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Time analysis
trips_df['pickup_hour'] = pd.to_datetime(trips_df['tpep_pickup_datetime']).dt.hour

plt.figure(figsize=(12, 5))
trips_df['pickup_hour'].value_counts().sort_index().plot(kind='bar')
plt.title('Trips by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Number of Trips')
plt.xticks(rotation=0)
plt.show()

## Zero Distance Analysis (Key FDE Judgment)

In [ ]:
# Detailed analysis of zero distance trips
zero_dist = trips_df[trips_df['trip_distance'] == 0].copy()

# Calculate duration
zero_dist['duration_minutes'] = (
    pd.to_datetime(zero_dist['tpep_dropoff_datetime']) - 
    pd.to_datetime(zero_dist['tpep_pickup_datetime'])
).dt.total_seconds() / 60

print(f"\nZero Distance Trips Analysis:")
print(f"Total: {len(zero_dist)}")
print(f"\nFare statistics:")
print(zero_dist['fare_amount'].describe())
print(f"\nDuration statistics:")
print(zero_dist['duration_minutes'].describe())

# How many would pass our validation?
valid_zero_dist = zero_dist[
    (zero_dist['duration_minutes'] >= 2) & 
    (zero_dist['fare_amount'] >= 2.50)
]
print(f"\nWould pass validation (duration >= 2 min AND fare >= $2.50): {len(valid_zero_dist)}")
print(f"Would fail validation: {len(zero_dist) - len(valid_zero_dist)}")

## Next Steps

1. Run the validation pipeline: `python src/validation/validators.py`
2. Build workflow model: `python src/modeling/workflow_model.py`
3. Calculate metrics: `python src/metrics/calculator.py`
4. Or run complete pipeline: `python src/pipeline/main_pipeline.py`